<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>R Analysis</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook characterizes PacBio read lengths and coverage for Phase 1 samples. It collects ComputeReadLengths histograms, joins extraction-platform metadata (Autogen vs Chemagen), and plots per-sample length curves plus coverage from ≥10 kb versus shorter reads.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
library(reticulate)
library(bigrquery)
library(ggplot2)
library(tidyverse)
#library(ggpubr)
#library(vcfR)

In [ ]:
# The FISS (FireCloud) Python API, called from R through reticulate, for reading Terra data tables.
fapi = import('firecloud.api')

In [ ]:
# Workspace details Terra sets as environment variables.
BILLING_PROJECT_ID = Sys.getenv('GOOGLE_PROJECT')       # the Google billing project
WORKSPACE_BUCKET = Sys.getenv('WORKSPACE_BUCKET')       # the workspace bucket
WORKSPACE_NAME = Sys.getenv('WORKSPACE_NAME')           # workspace name
WORKSPACE_NAMESPACE = Sys.getenv('WORKSPACE_NAMESPACE') # workspace namespace

In [ ]:
# Load a Terra data table into a tibble, converting 'true'/'false' list columns to logicals.
load_workspace_table <- function(table_name, workspace_namespace=WORKSPACE_NAMESPACE, workspace_name=WORKSPACE_NAME) {
    a = fapi$get_entities(WORKSPACE_NAMESPACE, WORKSPACE_NAME, table_name)

    json = a$json()
    df = tibble(metadata = json) %>% unnest_wider(metadata) %>% unnest_wider(attributes)

    for (name in df %>% select(where(is.list)) %>% names()) {
        #print(df[,name])
        q = as.vector(unlist(df[,name])) %in% c("true", "TRUE", "false", "FALSE") %>% unique()
        if (q) {
            df[,name] = as.logical(as.vector(unlist(df[,name])))
        }
    }
    
    return(df)
}

In [ ]:
# The per-sample table for the T2T-aligned HudsonAlpha samples.
ha_sample_t2t_df = load_workspace_table('ha-sample-t2t')

In [ ]:
ha_sample_t2t_df

In [ ]:
nrow(ha_sample_t2t_df)

In [ ]:
# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA <- Sys.getenv("AOU_LR_DATA")
if (AOU_LR_DATA == "") stop("Set the AOU_LR_DATA environment variable before running this notebook")

In [ ]:
# Sample selection sheet for the Broad 10k long-read set.
system(paste0("gsutil cp ", AOU_LR_DATA, "/broad_10k_sample_selection.csv broad_10k_sample_selection.csv"), intern=TRUE)


In [ ]:
broad10k_df = read_delim("broad_10k_sample_selection.csv", delim=",")

In [ ]:
broad10k_df

In [ ]:
# Download every sample's read-length list (output of the ComputeReadLengths workflow).
system("mkdir -p read_lengths read_lengths_v2/ComputeReadLengths")

In [ ]:
WORKSPACE_BUCKET

In [ ]:
system(paste0("gsutil ls -lh ", AOU_LR_DATA, "/outputs/GRCh38/ComputeReadLengths/"), intern=TRUE)

In [ ]:
#system(paste0("gsutil -m rsync -Cr ", AOU_LR_DATA, "/outputs/GRCh38/ComputeReadLengths/ read_lengths/ComputeReadLengths/"), intern=TRUE)
system(paste0("gsutil -m rsync -Cr ", AOU_LR_DATA, "/outputs/GRCh38/ComputeReadLengths/ read_lengths_v2/ComputeReadLengths/"), intern=TRUE)

In [ ]:
system("ls -l read_lengths_v2/ComputeReadLengths/ | wc -l", intern=TRUE)

In [ ]:
#full_df = inner_join(broad10k_df, inner_join(smrtcode_df, as_tibble(cbind(name = gsub(".read_lengths.txt.gz", "", rls), rls))))


In [ ]:
#full_df

In [ ]:
# Per-sample read-length files.
rls = system("ls read_lengths_v2/ComputeReadLengths/", intern=TRUE)

In [ ]:
rls

In [ ]:
length(rls)

In [ ]:
rls[1:3]

In [ ]:
# Histogram each sample's read lengths in 100 bp bins from 0 to 40 kbp (400 bins). df has one row per
# bin: column 1 is the bin start and each further column is one sample's counts.
df = cbind(seq(0, 39900, by=100))
df.names = c("bin")

for (rl in rls) {
    d = read_tsv(paste("read_lengths_v2/ComputeReadLengths/", rl, sep=""), col_names=c("rl"), show_col_types=FALSE)
    h = hist(d$rl, breaks=seq(0, max(d$rl) + 100, by=100), plot=FALSE)
    
    df = cbind(df, h$counts[1:400])
    df.names = c(df.names, rl)
}

colnames(df) = gsub(".read_lengths.txt.gz", "", df.names)

In [ ]:
#full_df %>% pull(Extraction) %>% unique()

In [ ]:
df

In [ ]:
print(df[, "example_sample"])

In [ ]:
# Sample manifests from HudsonAlpha (the sequencing centre): research IDs, and passing samples with their
# DNA extraction platform (Autogen or Chemagen).
if (!file.exists("ha_research_ids.csv")) {
    system(paste0("gsutil cp ", AOU_LR_DATA, "/ha_research_ids.csv ha_research_ids.csv"), intern=TRUE)
}

ha_df = read_delim("ha_research_ids.csv", delim=",", col_types = "ccc")
ha_df = ha_df %>% rename(name = Research_ID)

In [ ]:
if (!file.exists("ha_passing_sample.csv")) {
    system(paste0("gsutil cp ", AOU_LR_DATA, "/ha_passing_sample.csv ha_passing_sample.csv"), intern=TRUE)
}

ha_passing_samples_df = read_delim("ha_passing_sample.csv", delim=",")

In [ ]:
# Join the manifests to get each sample's extraction platform.
full_df = inner_join(ha_passing_samples_df, ha_df %>% mutate(BIOBANKID_SAMPLEID = paste(Biobank_ID, Sample_ID, sep="_")))


In [ ]:
full_df = full_df %>% rename(Extraction = `Extraction Platform`)

In [ ]:
full_df

In [ ]:
df

In [ ]:
# Read-length histograms coloured by extraction platform. Also computes, per sample, the bases in reads
# below and above 10 kbp (bin count x bin start) and the fraction of reads above 10 kbp.
par(mar=c(6, 6, 4, 2))
plot(0, 0, type="n", lwd=1, las=1, ylab="", xlim=c(0, 40000), ylim=c(0, 40000), xlab="Read length (bp)", bty="n", cex.axis=1.3, cex.lab=1.3)
mtext("Counts", 2, line=5, cex=1.3)
legend("topleft", c("Autogen", "Chemagen"), col=c("#0000ff", "#ff0000"), lwd=1)

bases_in_reads_below_10kbp = c()
bases_in_reads_above_10kbp = c()
frac_reads_above_10kbp = c()

for (i in 2:ncol(df)) {
     if (full_df %>% filter(name == colnames(df)[i]) %>% nrow() == 1) {
        extraction = (full_df %>% filter(name == colnames(df)[i]) %>% pull(Extraction))[1]
         
        color = "#00ff0019"
        if (extraction == "Autogen") {
            color = "#0000ff19"
        } else if (extraction == "Chemagen") {
            color = "#ff000019"
        }

        points(df[,c(1, i)], type="l", col=color)

        bases_in_reads_below_10kbp = c(
            bases_in_reads_below_10kbp,
            sum(df[1:100,1] * df[1:100,i])
        )
         
        bases_in_reads_above_10kbp = c(
            bases_in_reads_above_10kbp,
            sum(df[101:nrow(df),1] * df[101:nrow(df),i])
        )
        
        frac_reads_above_10kbp = c(
            frac_reads_above_10kbp,
            sum(df[101:nrow(df),i]) / sum(df[,i])
        )
    }
}

In [ ]:
# Scratch checks of the coverage arithmetic (3e9 is the approximate genome size). Cells 35-37 refer to
# variables (j, i) set elsewhere.
df[1:3,1]
df[1:3,2]
sum( df[,1] * df[,2] ) / 3e9

In [ ]:
df[,j]

In [ ]:
df[,1] * df[,j]

In [ ]:
sum(df[101:nrow(df),1] * df[101:nrow(df),i]) / (8*3e9)

In [ ]:
# Coverage contributed by reads >= 10 kbp and < 10 kbp; the red line marks 8x.
hist(bases_in_reads_above_10kbp / (3e9), las=1, breaks=seq(0, 20, by=0.41), probability = TRUE, cex.lab=1.3, cex.axis=1.3, main='HudsonAlpha', xlab='Coverage (>= 10 kbp reads)')
abline(v=8, col="red", lty=2)

In [ ]:
hist(bases_in_reads_below_10kbp / (3e9), las=1, breaks=seq(0, 20, by=1), probability=TRUE, cex.lab=1.3, cex.axis=1.3, main='HudsonAlpha', xlab='Coverage (< 10 kbp reads)')


In [ ]:
# Fraction of samples with less than 7.5x coverage from reads >= 10 kbp.
sum((bases_in_reads_above_10kbp / (3e9)) < 7.5) / length(bases_in_reads_above_10kbp)

In [ ]:
# Yield in reads >= 10 kbp relative to 23.5 Gbp.
hist(bases_in_reads_above_10kbp / (23.5e9), las=1, xlim=c(0, 2.0), breaks=seq(0, 3, by=0.05), probability=TRUE, cex.lab=1.3, cex.axis=1.3, main='HudsonAlpha', xlab=expression('Y'[10]*'/Y'[T]))


In [ ]:
mean(frac_reads_above_10kbp)

In [ ]:
# Scale a vector to the range 0-1.
min_max_norm <- function(x) {
    (x - min(x)) / (max(x) - min(x))
}

In [ ]:
# Min-max normalised read-length histograms, coloured by extraction platform, with the number of
# samples per platform.
par(mar=c(6, 6, 4, 2))
plot(0, 0, type="n", lwd=1, las=1, ylab="", xlim=c(0, 20000), ylim=c(0, 1.0), xlab="Read length (bp)", bty="n", cex.axis=1.3, cex.lab=1.3)
mtext("Normalized counts", 2, line=5, cex=1.3)

autogen = 0
chemagen = 0
other = 0

for (i in 2:ncol(df)) {
    if (full_df %>% filter(name == colnames(df)[i]) %>% nrow() == 1) {
        extraction = (full_df %>% filter(name == colnames(df)[i]) %>% pull(Extraction))[1]
        if (extraction == "Autogen") {
            color = "#0000ff19"
            autogen = autogen + 1
        } else {
            color = "#ff000019"
            chemagen = chemagen + 1
        }

        points(df[,1], min_max_norm(df[,i]), type="l", col=color)
    } else {
        other = other + 1
    }
}

print(autogen)
print(chemagen)
print(other)

In [ ]:
# The same with standardised (z-scored) axes.
par(mar=c(6, 6, 4, 2))
plot(0, 0, type="n", lwd=1, las=1, ylab="", xlim=c(-2, 2), ylim=c(-1, 3), xlab="Normalized read length (bp)", bty="n", cex.axis=1.3, cex.lab=1.3)
mtext("Normalized counts", 2, line=5, cex=1.3)

for (i in 2:ncol(df)) {
    if (full_df %>% filter(name == colnames(df)[i]) %>% nrow() == 1) {
        extraction = (full_df %>% filter(name == colnames(df)[i]) %>% pull(Extraction))[1]
        if (extraction == "Autogen") {
            color = "#0000ff19"
        } else {
            color = "#ff000019"
        }

        points(scale(df[,c(1, i)]), type="l", col=color)
    }
}

In [ ]:
# Normalised histograms for each extraction platform on its own plot.
par(mar=c(6, 6, 4, 2))
plot(0, 0, type="n", lwd=1, las=1, ylab="", xlim=c(0, 20000), ylim=c(0, 1.0), xlab="Read length (bp)", bty="n", cex.axis=1.3, cex.lab=1.3)
mtext("Normalized counts", 2, line=5, cex=1.3)

for (i in 2:ncol(df)) {
    if (full_df %>% filter(name == colnames(df)[i]) %>% nrow() == 1) {
        extraction = (full_df %>% filter(name == colnames(df)[i]) %>% pull(Extraction))[1]
        if (extraction == "Autogen") {
            points(df[,1], min_max_norm(df[,i]), type="l", col="#0000ff19")
        }
    }
}

par(mar=c(6, 6, 4, 2))
plot(0, 0, type="n", lwd=1, las=1, ylab="", xlim=c(0, 20000), ylim=c(0, 1.0), xlab="Read length (bp)", bty="n", cex.axis=1.3, cex.lab=1.3)
mtext("Normalized counts", 2, line=5, cex=1.3)

for (i in 2:ncol(df)) {
    if (full_df %>% filter(name == colnames(df)[i]) %>% nrow() == 1) {
        extraction = (full_df %>% filter(name == colnames(df)[i]) %>% pull(Extraction))[1]
        if (extraction == "Chemagen") {
            points(df[,1], min_max_norm(df[,i]), type="l", col="#ff000019")
        }
    }
}

In [ ]:
# Normal Q-Q plots of individual samples' histogram counts.
q = 13
qqnorm(df[,q], pch = 1, frame = FALSE)
qqline(df[,q], col = "steelblue", lwd = 2)

In [ ]:
library(car)

In [ ]:
qqPlot(df[,5])
plot(df[,1], df[,5])

In [ ]:
df